# fin-anim-agent — whiteboard icon generator (free Colab GPU)

Generates whiteboard-marker-style doodle PNGs for the `whiteboard_explainer` scene's
`"image"` visual mode, using a local Stable Diffusion model on Colab's free T4 GPU —
no paid API/MCP credits.

**Before running:** `Runtime` menu -> `Change runtime type` -> Hardware accelerator = **T4 GPU** (or better).

**After running:** download `whiteboard_icons.zip` (last cell), unzip it, and copy the PNGs
into this repo's `assets/whiteboard_icons/` folder. Then reference one in a beat:

```json
{"visual": "image", "image_path": "assets/whiteboard_icons/piggy_bank.png", "label": "Save $100", ...}
```

Results from a diffusion model are not deterministic and not guaranteed to look good on the
first try — re-run the generation cell (or tweak `PROMPT_TEMPLATE`/`NEGATIVE_PROMPT`) and
regenerate the ones that came out badly before downloading.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

assert torch.cuda.is_available(), "No GPU — set Runtime > Change runtime type > T4 GPU, then re-run"

MODEL_ID = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16, safety_checker=None)
pipe = pipe.to("cuda")
print("Model loaded on", pipe.device)

In [ ]:
# Concept per icon name — keep these keys in sync with WHITEBOARD_ICONS in
# skills/fin-anim/scripts/schema.py. Add/remove entries here to generate a
# different or larger icon set.
ICON_CONCEPTS = {
    "dollar": "a dollar sign symbol",
    "arrow_up": "a single upward pointing arrow",
    "arrow_down": "a single downward pointing arrow",
    "clock": "a simple round clock face with an hour hand and a minute hand",
    "house": "a simple house with a triangular roof and a door",
    "piggy_bank": "a piggy bank with a coin slot on top",
    "coin_stack": "a stack of three round coins",
    "calculator": "a calculator with a screen and a grid of buttons",
    "chart_bar": "a simple bar chart with four vertical bars of different heights",
    "lightbulb": "a lightbulb with a few small rays of light around it",
    "scale": "a balance scale with two hanging pans",
}

PROMPT_TEMPLATE = (
    "{concept}, simple black ink line drawing, whiteboard marker sketch style, "
    "thick clean bold outlines, minimalist doodle icon, plain white background, "
    "no shading, no color, no gradient, hand-drawn"
)
NEGATIVE_PROMPT = (
    "photo, photorealistic, 3d render, color, shading, gradient, text, watermark, "
    "signature, blurry, complex background, multiple objects, realistic"
)

print(f"{len(ICON_CONCEPTS)} icons queued: {list(ICON_CONCEPTS)}")

In [ ]:
import os

OUT_DIR = "whiteboard_icons_raw"
os.makedirs(OUT_DIR, exist_ok=True)

generator = torch.Generator("cuda").manual_seed(0)

for name, concept in ICON_CONCEPTS.items():
    prompt = PROMPT_TEMPLATE.format(concept=concept)
    image = pipe(
        prompt,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=generator,
    ).images[0]
    image.save(os.path.join(OUT_DIR, f"{name}.png"))
    print("saved", name)

## Make the white background transparent

Simple RGB-threshold cutout (near-white -> transparent) — good enough for a flat
whiteboard-style doodle on a plain background. If a specific icon has a messy
background (diffusion models don't always give perfectly flat white), lower
`WHITE_THRESHOLD` for that one and re-run, or touch it up manually before download.

In [ ]:
from PIL import Image
import numpy as np

FINAL_DIR = "whiteboard_icons"
os.makedirs(FINAL_DIR, exist_ok=True)
WHITE_THRESHOLD = 235  # 0-255; lower = cuts out more near-white pixels

for name in ICON_CONCEPTS:
    img = Image.open(os.path.join(OUT_DIR, f"{name}.png")).convert("RGBA")
    arr = np.array(img)
    is_near_white = np.all(arr[:, :, :3] >= WHITE_THRESHOLD, axis=-1)
    arr[is_near_white, 3] = 0
    Image.fromarray(arr).save(os.path.join(FINAL_DIR, f"{name}.png"))
    print("cut out background:", name)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("whiteboard_icons", "zip", FINAL_DIR)
files.download("whiteboard_icons.zip")